#### Física de la Radioterapia. Máster de Física Biomédica. Universidad Complutense de Madrid
# `OpenTPS` en **Colab**
## Cálculo de un tratamiento simple de próstata con haces de **fotones**
-----

Este cuaderno sirve de plantilla de ejemplo de cómo diseñar un tratamiento, optimizarlo y calcular su distribución de dosis asociada mediante `OpenTPS`.


> La instalación en Colab es todavía experimental y requiere utilizar pip sobre la distribución del paquete. No hemos conseguido instalarla utilizando **uv** por problemas de compatibilidad entre las versiones de `numpy` y `scipy`

> El entorno de ejecución puede ser basado en CPU o en GPU. El uso de cupy en `OpenTPS` (entornos GPU`) es experimental. No hemos conseguido las ganancias de velocidad de ejecución de las que habla la documentación de `OpenTPS` que podrían ser hasta de 18X.

In [ ]:
import sys

if "google.colab" in sys.modules:
  from IPython import get_ipython
  get_ipython().system('git clone https://gitlab.com/openmcsquare/opentps.git')
  get_ipython().system('pip install ./opentps')
  output = get_ipython().system('nvidia-smi -L')

  # Ensure output_str is always a string, even if output is None
  if output is None:
    output_str = ""
  else:
    output_str = output.decode('utf-8') if isinstance(output, bytes) else output

  if "No devices found" in output_str:
    print("El entorno de ejecución es CPU.")
  else:
    get_ipython().system('uv pip install cupy-cuda12x')


Importación de módulos generales

In [ ]:
import os
import logging
import numpy as np
from matplotlib import pyplot as plt
import sys
import copy
from scipy.sparse import csc_matrix

Importación de módulos de `OpenTPS`

In [ ]:
from opentps.core.io.dataLoader import readData
from opentps.core.io.dicomIO import writeRTDose, writeDicomCT, writeRTPlan, writeRTStruct
from opentps.core.processing.planOptimization.tools import evaluateClinical
from opentps.core.data.images import CTImage
from opentps.core.data.images import ROIMask
from opentps.core.data import DVH
from opentps.core.data import Patient
import opentps.core.processing.planOptimization.objectives.dosimetricObjectives as doseObj
from opentps.core.io.scannerReader import readScanner
from opentps.core.io.serializedObjectIO import saveRTPlan, loadRTPlan
from opentps.core.processing.doseCalculation.doseCalculationConfig import DoseCalculationConfig
from opentps.core.processing.imageProcessing.resampler3D import resampleImage3DOnImage3D
from opentps.core.processing.planOptimization.planOptimization import  IntensityModulationOptimizer
from opentps.core.processing.doseCalculation.photons.cccDoseCalculator import CCCDoseCalculator
from opentps.core.data.plan import PhotonPlanDesign

## Estudio del paciente y configuración del planificador

Descargar y cargar los datos del paciente de próstata

In [ ]:
%%bash
# Ejecutar código Python para descargar los datos del Drive
python << EOF > /dev/null 2>&1
# Descargar los datos compartidos de Google Drive
import gdown
url = 'https://drive.google.com/file/d/1iB2dlt6QKEcI_n92mv9jMbwqy4ZuMKjn/view?usp=sharing'
output = '/content/Prostate.zip'
gdown.download(url, output, quiet=True, fuzzy=True)
EOF

# Extracción de los datos para el cálculo de dosis de ejemplo

# Descomprimir archivo de datos
unzip /content/Prostate.zip -d /content/ >/dev/null

# Eliminar el archivo comprimido
rm -f /content/Prostate.zip

In [ ]:
patientDataPath = "/content/PatientID/"
patientData = readData(patientDataPath)

In [ ]:
# Escáner de simulación
ct = patientData[1]

# Estructuras de interés delimitadas
rt_struct= patientData[0]

#  Listar los nombres de los ROI y los volúmenes objetivo
rt_struct.print_ROINames()


Definición del volumen blanco y los órganos de riesgo

In [ ]:
# Definir el volumen objetivo
target = rt_struct.getContourByName('PTV prostata').getBinaryMask(origin=ct.origin,gridSize=ct.gridSize,spacing=ct.spacing)

# Definir los órganos de riesgo relevantes
OAR_rectum = rt_struct.getContourByName("Recto").getBinaryMask(origin=ct.origin,gridSize=ct.gridSize,spacing=ct.spacing)
OAR_bladder = rt_struct.getContourByName("Vejiga").getBinaryMask(origin=ct.origin,gridSize=ct.gridSize,spacing=ct.spacing)
BODY = rt_struct.getContourByName("BODY").getBinaryMask(origin=ct.origin,gridSize=ct.gridSize,spacing=ct.spacing)

## Diseño del plan
-----------
Para mantener simple la plantilla de ejemplo diseñamos un plan con un único campo.

In [ ]:
beamNames = ["Beam1"]
gantryAngles = [90.]
couchAngles = [0.]

## Generación cáculo del haz
----

In [ ]:
# Sobreescribir el comportamiento de stdout y stderr para recibir inmediantamente
# la información de ejecución del cálculo
class FlushingStream:
    def __init__(self, stream):
        self.stream = stream

    def write(self, data):
        self.stream.write(data)
        self.stream.flush()

    def __getattr__(self, attr):
        return getattr(self.stream, attr)

sys.stdout = FlushingStream(sys.stdout)
sys.stderr = FlushingStream(sys.stderr)

In [ ]:
## Dose computation from plan
ccc = CCCDoseCalculator(batchSize= 10)
logger.info('Algoritmo de cálculo configurado')
ccc.ctCalibration = readScanner(DoseCalculationConfig().scannerFolder)
logger.info('Configuración Escáner CT leída')

# Load / Generate new plan
output_path = '/content/output'
get_ipython().system('mkdir -p ' + output_path)
plan_file = os.path.join(output_path, "PhotonPlan_WaterPhantom_cropped_resampled.tps")

if os.path.isfile(plan_file): ### Remove the False to load the plan
    plan = loadRTPlan(plan_file, radiationType='photon')
    logger.info('Plan loaded')
    logger.info(f'plan.planDesign.beamlets._doseGrid: {plan.planDesign.beamlets._gridSize}')
else:
    logger.info('El plan no se ha definido previamente. Diseñar el plan...')
    planDesign = PhotonPlanDesign()
    planDesign.ct = ct
    planDesign.targetMask = target
    planDesign.isocenterPosition_mm = None # None take the center of mass of the target
    planDesign.gantryAngles = gantryAngles
    planDesign.couchAngles = couchAngles
    planDesign.beamNames = beamNames
    planDesign.calibration = ctCalibration
    planDesign.xBeamletSpacing_mm = 25
    planDesign.yBeamletSpacing_mm = 25
    planDesign.targetMargin = 2.5
    planDesign.defineTargetMaskAndPrescription(target = target, targetPrescription = 20.)

    plan = planDesign.buildPlan()
    logger.info('Plan diseñado!')

    logger.info('Calcular beamlets...')
    beamlets = ccc.computeBeamlets(ct, plan)
    doseInfluenceMatrix = copy.deepcopy(beamlets)
    logger.info('Beamlets calculados!')

    plan.planDesign.beamlets = beamlets
    logger.info('Beamlets cargados.')
    beamlets.storeOnFS(os.path.join(output_path, "BeamletMatrix_" + plan.seriesInstanceUID + ".blm"))
    logger.info('Beamlets almacenados en el Sistema de archivos')
    # Save plan with initial spot weights in serialized format (OpenTPS format)
    saveRTPlan(plan, plan_file)
    logger.info('Plan creado y guardado en el sistema de archivos.')